# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR⁲](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nPublished: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs in the dataset.

**Note**: All schema objects are referenced by their `@id`. Use the following cell to enumerate record sets and their fields.

In [ ]:
# Helper function: extract record sets, fields, and columns by @id
# The Croissant metadata structure

def show_recordsets_fields(dataset):
    print("\nAvailable Record Sets and their fields/columns:\n")
    record_sets = dataset.metadata.recordSet
    rs_info = {}
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']} - {rs.get('name', '')}")
        rs_info[rs['@id']] = { 'fields': [], 'columns': [] }
        for field in rs.get('field', []):
            print(f"   Field: {field['@id']}: {field.get('name', '')} | dataType: {field.get('dataType', '')}")
            rs_info[rs['@id']]['fields'].append(field['@id'])
        for col in rs.get('column', []):
            print(f"   Column: {col['@id']}: {col.get('name', '')} | source: {col.get('source', '')}")
            rs_info[rs['@id']]['columns'].append(col['@id'])
        print()
    return rs_info

rs_info = show_recordsets_fields(dataset)

# For demonstration in subsequent cells, let's select the first available record set:
record_sets_list = list(rs_info.keys())
print(f"Discovered record set @ids: {record_sets_list}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the previous overview.

You can reference fields and columns by their `@id` for extraction and processing.

In [ ]:
# Extract data from each record set

# If there are no record sets, the below code will elegantly skip.
dataframes = {}
if record_sets_list:
    for record_set in record_sets_list:
        try:
            records = list(dataset.records(record_set=record_set))
            if records:
                dataframes[record_set] = pd.DataFrame(records)
                print(f"Loaded {len(records)} records from RecordSet {record_set}")
            else:
                print(f"No records found for RecordSet {record_set}")
        except Exception as e:
            print(f"Could not extract records for {record_set} due to: {e}")
    # Select first record set for demonstration
    if dataframes:
        main_record_set_id = list(dataframes.keys())[0]
        print(f"\nDataFrame columns for record set {main_record_set_id}:")
        print(dataframes[main_record_set_id].columns.tolist())
        display(dataframes[main_record_set_id].head())
    else:
        main_record_set_id = None
        print("No dataframes were created.")
else:
    main_record_set_id = None
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data using the schema. Operations like removing outliers, transforming data, or grouping by fields can be performed at this stage.

In [ ]:
# Example EDA: Filter and normalize a numeric field in the main record set
if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Attempt to auto-detect a numeric field (e.g., containing 'abundance', 'count', 'coverage', 'weight', etc.)
    possible_numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]
    if not possible_numeric_fields:
        for c in df.columns:
            # Try converting
            try:
                df[c] = pd.to_numeric(df[c], errors='ignore')
            except Exception:
                pass
        possible_numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

    print(f"Possible numeric fields: {possible_numeric_fields}")
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"\nOperating on numeric field: {numeric_field}\n")

        # Filter: Show only rows where the numeric field > its median (as a simple threshold)
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()

        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Group by first non-numeric field (if any)
        group_fields = [c for c in df.columns if c not in possible_numeric_fields]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            group_field = None
    else:
        numeric_field = None
        group_field = None
        print("No suitable numeric field found for EDA.")
else:
    numeric_field = None
    group_field = None
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field is not None:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field detected, make boxplot
    if group_field is not None and group_field in df:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field].astype(str), y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
This notebook demonstrated loading, schema introspection, and basic exploration of the FAIR⁲ dataset using the `mlcroissant` library. You can adjust field and record set `@id` references and analysis logic for deeper investigation of this and similar Croissant-based datasets.